# 05 — Schema evolution: absorb the v2 keys

The alert fired. The data team triages, decides the three new keys are real and useful, and adds them to the contract. Two steps:

1. **Widen the allow-list** (`known_payload_keys`) so the drift detector goes back to green
2. **Widen `silver.plays`** to add the three new typed columns, then back-fill from bronze

This is the moment to point out:

- Bronze never had to be rewritten — the VARIANT payload absorbed v2 transparently
- The drift detector tells you *what* to add to silver, not just "something changed"
- Delta `mergeSchema=true` widens the table in place; existing rows get `NULL` for new columns
- Lakehouse Monitoring picks up the new columns on the next refresh

In [0]:
%pip install dotenv

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os
from dotenv import load_dotenv
load_dotenv()

try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

UC_CATALOG     = os.getenv('UC_CATALOG', 'alexander_booth')
UC_SCHEMA      = os.getenv('UC_SCHEMA',  'hockey_schema_monitoring')
BRONZE_TABLE   = f'{UC_CATALOG}.{UC_SCHEMA}_bronze.plays_raw'
SILVER_TABLE   = f'{UC_CATALOG}.{UC_SCHEMA}_silver.plays'
KNOWN_KEYS_TBL = f'{UC_CATALOG}.{UC_SCHEMA}_silver.known_payload_keys'
DRIFT_VIEW     = f'{UC_CATALOG}.{UC_SCHEMA}_silver.v_unknown_payload_keys'

## Step 1 — Widen the allow-list

In [0]:
NEW_KEYS = ['expected_goals', 'shot_quality_index', 'puck_speed_mph']

from pyspark.sql import Row
(spark.createDataFrame([Row(key=k) for k in NEW_KEYS])
      .write.mode('append').saveAsTable(KNOWN_KEYS_TBL))

spark.sql(f'SELECT * FROM {KNOWN_KEYS_TBL} ORDER BY key').show(truncate=False)

+------------------+
|key               |
+------------------+
|event_ts          |
|event_type        |
|expected_goals    |
|game_id           |
|period            |
|period_time       |
|player_id         |
|puck_speed_mph    |
|season            |
|shot_quality_index|
|team_abbrev       |
|x_coord           |
|y_coord           |
+------------------+



## Step 2 — Widen silver and back-fill

We rebuild `silver.plays` from bronze with the new columns selected out. Because the table was created with `delta.columnMapping.mode = 'name'` and we write with `mergeSchema=true`, this is an in-place schema widen — the table identity, history, and any downstream consumers of unchanged columns keep working.

In [0]:
from pyspark.sql import functions as F

bronze = spark.read.table(BRONZE_TABLE)
silver = (bronze.selectExpr(
    'ingest_ts',
    'source_version',
    "payload:game_id::string      AS game_id",
    "payload:season::string       AS season",
    "payload:period::int          AS period",
    "payload:period_time::string  AS period_time",
    "payload:event_type::string   AS event_type",
    "payload:team_abbrev::string  AS team_abbrev",
    "payload:player_id::long      AS player_id",
    "payload:x_coord::int         AS x_coord",
    "payload:y_coord::int         AS y_coord",
    "cast(payload:event_ts::string AS timestamp) AS event_ts",
    # the three new v2 columns — NULL for v1 rows, populated for v2
    "payload:expected_goals::double      AS expected_goals",
    "payload:shot_quality_index::double  AS shot_quality_index",
    "payload:puck_speed_mph::double      AS puck_speed_mph",
))

(silver.write
       .mode('overwrite')
       .option('overwriteSchema', 'true')
       .saveAsTable(SILVER_TABLE))

spark.sql(f'DESCRIBE TABLE {SILVER_TABLE}').show(truncate=False)

+------------------+---------+-------+
|col_name          |data_type|comment|
+------------------+---------+-------+
|ingest_ts         |timestamp|NULL   |
|source_version    |string   |NULL   |
|game_id           |string   |NULL   |
|season            |string   |NULL   |
|period            |int      |NULL   |
|period_time       |string   |NULL   |
|event_type        |string   |NULL   |
|team_abbrev       |string   |NULL   |
|player_id         |bigint   |NULL   |
|x_coord           |int      |NULL   |
|y_coord           |int      |NULL   |
|event_ts          |timestamp|NULL   |
|expected_goals    |double   |NULL   |
|shot_quality_index|double   |NULL   |
|puck_speed_mph    |double   |NULL   |
+------------------+---------+-------+



In [0]:
print('Coverage by source_version (new columns NULL for v1, populated for v2):')
spark.sql(f'''
    SELECT
        source_version,
        COUNT(*)                                                  AS rows,
        ROUND(AVG(CASE WHEN expected_goals     IS NOT NULL THEN 1 ELSE 0 END), 3) AS pct_with_xg,
        ROUND(AVG(CASE WHEN shot_quality_index IS NOT NULL THEN 1 ELSE 0 END), 3) AS pct_with_sqi,
        ROUND(AVG(CASE WHEN puck_speed_mph     IS NOT NULL THEN 1 ELSE 0 END), 3) AS pct_with_speed
    FROM {SILVER_TABLE}
    GROUP BY source_version
    ORDER BY source_version
''').show()

Coverage by source_version (new columns NULL for v1, populated for v2):
+--------------+----+-----------+------------+--------------+
|source_version|rows|pct_with_xg|pct_with_sqi|pct_with_speed|
+--------------+----+-----------+------------+--------------+
|            v1|3200|        0.0|         0.0|           0.0|
|            v2| 400|        1.0|         1.0|           1.0|
+--------------+----+-----------+------------+--------------+



In [0]:
print('Drift detector should now be empty:')
spark.sql(f'SELECT * FROM {DRIFT_VIEW}').show(truncate=False)
print('\nRefresh the DBSQL Alert in the UI — it should flip back to OK.')

Drift detector should now be empty:
+-----------+---------+----------+---------+------------------------+---------------+
|payload_key|sightings|first_seen|last_seen|source_versions_affected|source_versions|
+-----------+---------+----------+---------+------------------------+---------------+
+-----------+---------+----------+---------+------------------------+---------------+


Refresh the DBSQL Alert in the UI — it should flip back to OK.


In [0]:
# Trigger a Lakehouse Monitoring refresh so the dashboard picks up the new columns.
refresh = w.lakehouse_monitors.run_refresh(SILVER_TABLE)
print(f'LHM refresh queued: id={refresh.refresh_id} state={refresh.state}')

LHM refresh queued: id=749028031869250 state=MonitorRefreshInfoState.RUNNING
